# Spatiotemporal Analysis of LULC Change and Property-Value Growth in Galougah

This notebook contains the spatial and econometric analyses used to examine urban expansion, land-use transitions, green infrastructure, and property-value growth in Galougah, Mazandaran.

## 1. Setup

The required libraries are loaded here. Input and output files are assumed to be stored in the same project directory as the notebook.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import statsmodels.api as sm
import statsmodels.formula.api as smf

from rasterio.features import shapes
from shapely.geometry import MultiPolygon, Point, shape
from shapely.ops import unary_union
from sklearn.neighbors import NearestNeighbors
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

PROJECT_DIR = Path.cwd()

REGRESSION_FILE = PROJECT_DIR / "galougah_ready_for_regression.xlsx"
RASTER_FILE = PROJECT_DIR / "LULC_2030_StateOfTheArt_Predicted.tif"
FRONTIER_FILE = PROJECT_DIR / "urban_expansion_frontier.shp"
POLYGON_FILE = PROJECT_DIR / "urban_expansion_polygon.shp" 

## 2. Urban expansion frontier

Built-up pixels in the 2030 LULC raster are converted to vector geometries and merged. The boundary of the resulting built-up area is used as the urban expansion frontier.

In [ ]:
built_up_class = 1

with rasterio.open(RASTER_FILE) as src:
    lulc = src.read(1)
    transform = src.transform
    raster_crs = src.crs

built_mask = lulc == built_up_class
vector_shapes = shapes(lulc, mask=built_mask, transform=transform)

built_patches = [
    shape(geom)
    for geom, value in vector_shapes
    if value == built_up_class
]

if not built_patches:
    raise ValueError("No built-up pixels were found for the selected class.")

built_area = unary_union(MultiPolygon(built_patches))
frontier = built_area.boundary

frontier_gdf = gpd.GeoDataFrame(
    geometry=[frontier], crs=raster_crs
).to_crs("EPSG:32639")

built_area_gdf = gpd.GeoDataFrame(
    geometry=[built_area], crs=raster_crs
).to_crs("EPSG:32639")

frontier_gdf.to_file(FRONTIER_FILE)
built_area_gdf.to_file(POLYGON_FILE)

## 3. Urban expansion metrics

Area and boundary length are calculated in the projected coordinate system and exported for use in the results tables.

In [ ]:
area_m2 = built_area_gdf.geometry.area.sum()
length_m = frontier_gdf.geometry.length.sum()

urban_metrics = pd.DataFrame(
    {
        "Metric": [
            "Raster file",
            "Built-up class",
            "Built-up patch count",
            "Urban expansion area (m2)",
            "Urban expansion area (km2)",
            "Urban frontier length (m)",
            "Urban frontier length (km)",
        ],
        "Value": [
            RASTER_FILE.name,
            built_up_class,
            len(built_patches),
            area_m2,
            area_m2 / 1e6,
            length_m,
            length_m / 1e3,
        ],
    }
)

urban_metrics.to_excel(
    PROJECT_DIR / "urban_expansion_metrics.xlsx",
    index=False,
)

urban_metrics

## 4. Distance to the urban frontier

Block-centre coordinates are projected to UTM Zone 39N. The minimum Euclidean distance from each block centre to the urban frontier is then calculated in kilometres.

In [ ]:
block_data = pd.read_excel(REGRESSION_FILE, sheet_name="Sheet1")

geometry = [
    Point(x, y)
    for x, y in zip(block_data["longitude"], block_data["latitude"])
]

blocks_gdf = gpd.GeoDataFrame(
    block_data.copy(),
    geometry=geometry,
    crs="EPSG:4326",
).to_crs("EPSG:32639")

frontier_gdf = gpd.read_file(FRONTIER_FILE).to_crs("EPSG:32639")
frontier_union = frontier_gdf.geometry.unary_union

blocks_gdf["distance_to_frontier"] = blocks_gdf.geometry.apply(
    lambda point: point.distance(frontier_union) / 1000
)

blocks_gdf[
    ["Block_ID", "Year", "distance_to_frontier"]
].head()

## 5. Frontier proximity plots

The following plots show the relationship between frontier distance, residential transaction price, and green-infrastructure coverage at the 500 m scale.

In [ ]:
blocks_gdf["log_price"] = np.log(
    blocks_gdf["Residential_IRR-real deal"].replace(0, np.nan)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=150)

price_data = blocks_gdf[
    ["distance_to_frontier", "log_price"]
].dropna()

axes[0].scatter(
    price_data["distance_to_frontier"],
    price_data["log_price"],
    alpha=0.7,
)

if len(price_data) >= 2:
    price_fit = np.polyfit(
        price_data["distance_to_frontier"],
        price_data["log_price"],
        1,
    )
    x_price = np.linspace(
        price_data["distance_to_frontier"].min(),
        price_data["distance_to_frontier"].max(),
        100,
    )
    axes[0].plot(x_price, np.polyval(price_fit, x_price))

axes[0].set_xlabel("Distance to urban expansion frontier (km)")
axes[0].set_ylabel("Log residential transaction price")
axes[0].set_title("Frontier distance and residential price")

gi_data = blocks_gdf[
    ["distance_to_frontier", "gi_buffer_500"]
].dropna()

axes[1].scatter(
    gi_data["distance_to_frontier"],
    gi_data["gi_buffer_500"],
    alpha=0.7,
)

if len(gi_data) >= 2:
    gi_fit = np.polyfit(
        gi_data["distance_to_frontier"],
        gi_data["gi_buffer_500"],
        1,
    )
    x_gi = np.linspace(
        gi_data["distance_to_frontier"].min(),
        gi_data["distance_to_frontier"].max(),
        100,
    )
    axes[1].plot(x_gi, np.polyval(gi_fit, x_gi))

axes[1].set_xlabel("Distance to urban expansion frontier (km)")
axes[1].set_ylabel("Green infrastructure proportion (500 m)")
axes[1].set_title("Frontier distance and green infrastructure")

plt.tight_layout()
plt.savefig(
    PROJECT_DIR / "frontier_distance_relationships.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 6. Regression dataset

Spatial indicators are merged with the land-use transition records by block and year. The main variables are cleaned and log-transformed where required.

In [ ]:
spatial_data = pd.read_excel(
    REGRESSION_FILE,
    sheet_name="Sheet1",
)

transition_data = pd.read_excel(
    REGRESSION_FILE,
    sheet_name="Land changes and growth",
)

transition_data.columns = transition_data.columns.str.strip()

spatial_data["block_num"] = (
    spatial_data["Block_ID"]
    .str.extract(r"(\d+)")
    .astype(int)
)

merged_data = pd.merge(
    spatial_data,
    transition_data,
    left_on=["block_num", "Year"],
    right_on=["block No", "year"],
    how="inner",
)

merged_data["growth_num"] = pd.to_numeric(
    merged_data["value growth percentage"],
    errors="coerce",
)

merged_data = merged_data.rename(
    columns={
        "Residential_IRR-real deal": "res_price",
        "change from": "change_from",
        "change to": "change_to",
    }
)

required_columns = [
    "longitude",
    "latitude",
    "growth_num",
    "gi_buffer_1000",
    "res_price",
]

merged_data = (
    merged_data
    .dropna(subset=required_columns)
    .copy()
)

merged_data["log_res_price"] = np.log1p(
    merged_data["res_price"]
)

merged_data["log_buffer"] = np.log1p(
    merged_data["gi_buffer_1000"]
)

merged_data["res_price_billion"] = (
    merged_data["res_price"] / 1e9
)

print(f"Valid observations: {len(merged_data)}")

## 7. Spatial lag

A five-nearest-neighbour structure is constructed from the block coordinates. Mean neighbouring value growth is used as the spatial-lag variable.

In [ ]:
coordinates = merged_data[
    ["longitude", "latitude"]
].to_numpy()

knn = NearestNeighbors(
    n_neighbors=5,
    algorithm="ball_tree",
).fit(coordinates)

distances, neighbour_indices = knn.kneighbors(coordinates)

merged_data["spatial_lag_growth"] = [
    merged_data["growth_num"].iloc[index].mean()
    for index in neighbour_indices
]

merged_data[
    ["Block_ID", "Year", "growth_num", "spatial_lag_growth"]
].head()

## 8. Regression models

Three specifications are estimated: an original-scale model, a log-covariate spatial model, and the same spatial model with HC1 robust standard errors.

In [ ]:
formula_1 = (
    "growth_num ~ "
    "C(change_from) + C(change_to) + "
    "gi_buffer_1000 + res_price"
)

formula_2 = (
    "growth_num ~ "
    "C(change_from) + C(change_to) + "
    "log_buffer + log_res_price + spatial_lag_growth"
)

model_1 = smf.ols(
    formula_1,
    data=merged_data,
).fit()

model_2 = smf.ols(
    formula_2,
    data=merged_data,
).fit()

model_3 = smf.ols(
    formula_2,
    data=merged_data,
).fit(cov_type="HC1")

print(model_3.summary())

## 9. Regression table

Coefficients and standard errors from the three model specifications are combined in a single table. Significance levels are reported using conventional stars.

In [ ]:
def significance_stars(p_value):
    if p_value < 0.01:
        return "***"
    if p_value < 0.05:
        return "**"
    if p_value < 0.10:
        return "*"
    return ""


models = {
    "Model (1)": model_1,
    "Model (2)": model_2,
    "Model (3)": model_3,
}

parameters = sorted(
    {
        parameter
        for model in models.values()
        for parameter in model.params.index
        if parameter != "Intercept"
    }
)

table_rows = []
row_labels = []

for variable in ["Intercept"] + parameters:
    coefficient_row = {}
    standard_error_row = {}

    for model_name, model in models.items():
        if variable in model.params.index:
            coefficient = model.params[variable]
            standard_error = model.bse[variable]
            p_value = model.pvalues[variable]

            coefficient_row[model_name] = (
                f"{coefficient:.4f}"
                f"{significance_stars(p_value)}"
            )
            standard_error_row[model_name] = (
                f"({standard_error:.4f})"
            )
        else:
            coefficient_row[model_name] = ""
            standard_error_row[model_name] = ""

    table_rows.extend(
        [coefficient_row, standard_error_row]
    )
    row_labels.extend(
        [variable, f"{variable}_se"]
    )

table_rows.append(
    {
        name: f"{model.rsquared:.4f}"
        for name, model in models.items()
    }
)
row_labels.append("R-squared")

table_rows.append(
    {
        name: f"{int(model.nobs):,}"
        for name, model in models.items()
    }
)
row_labels.append("Observations")

regression_table = pd.DataFrame(
    table_rows,
    index=row_labels,
)

regression_table.to_excel(
    PROJECT_DIR / "phase4_regression_table.xlsx"
)

regression_table

## 10. Spatial growth plot

Value growth is plotted against the log-transformed 1,000 m green-infrastructure measure, with residential price represented by the point colour scale.

In [ ]:
plt.figure(figsize=(10, 6))

scatter = plt.scatter(
    merged_data["log_buffer"],
    merged_data["growth_num"],
    c=merged_data["log_res_price"],
    s=70,
    alpha=0.8,
)

plt.colorbar(
    scatter,
    label="Log residential transaction price",
)

plt.xlabel("Log green infrastructure buffer (1,000 m)")
plt.ylabel("Value growth percentage")
plt.title("Green infrastructure and value growth")
plt.grid(True, linestyle="--", alpha=0.5)

plt.savefig(
    PROJECT_DIR / "phase4_spatial_growth_plot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 11. Raw-covariate spatial model

The raw-scale spatial specification is estimated separately. Actual values, fitted values, and residuals are exported for block-level comparison.

In [ ]:
formula_raw_spatial = (
    "growth_num ~ "
    "C(change_from) + C(change_to) + "
    "gi_buffer_1000 + res_price + spatial_lag_growth"
)

raw_spatial_model = smf.ols(
    formula_raw_spatial,
    data=merged_data,
).fit()

prediction_data = pd.DataFrame(
    {
        "Block_ID": merged_data["Block_ID"].to_numpy(),
        "Year": merged_data["Year"].to_numpy(),
        "Actual_Growth": merged_data["growth_num"].to_numpy(),
        "Predicted_Growth": raw_spatial_model.fittedvalues.to_numpy(),
        "Residuals": raw_spatial_model.resid.to_numpy(),
        "Real_Price_Billion_IRR": merged_data[
            "res_price_billion"
        ].to_numpy(),
        "GI_Buffer_1000": merged_data[
            "gi_buffer_1000"
        ].to_numpy(),
    }
)

prediction_data.to_excel(
    PROJECT_DIR / "phase5_model_predictions.xlsx",
    index=False,
)

prediction_data.head()

## 12. Actual and predicted growth

Observed and fitted values are compared with a 45-degree reference line to show the in-sample fit of the raw-covariate spatial model.

In [ ]:
plt.figure(figsize=(10, 6))

scatter = plt.scatter(
    prediction_data["Actual_Growth"],
    prediction_data["Predicted_Growth"],
    c=prediction_data["Real_Price_Billion_IRR"],
    s=70,
    alpha=0.8,
)

plt.colorbar(
    scatter,
    label="Residential transaction price (billion IRR)",
)

lower = min(
    prediction_data["Actual_Growth"].min(),
    prediction_data["Predicted_Growth"].min(),
)

upper = max(
    prediction_data["Actual_Growth"].max(),
    prediction_data["Predicted_Growth"].max(),
)

plt.plot(
    [lower, upper],
    [lower, upper],
    linestyle="--",
    linewidth=2,
    label="1:1 line",
)

plt.xlabel("Actual value growth percentage")
plt.ylabel("Predicted value growth percentage")
plt.title("Actual and predicted value growth")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)

plt.savefig(
    PROJECT_DIR / "phase5_actual_vs_predicted.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 13. Model diagnostics

Breusch–Pagan, Moran's I, and variance inflation factors are used to assess heteroskedasticity, residual spatial autocorrelation, and multicollinearity.

In [ ]:
diagnostic_model = smf.ols(
    formula_2,
    data=merged_data,
).fit()

residuals = diagnostic_model.resid.to_numpy()

bp_test = het_breuschpagan(
    residuals,
    diagnostic_model.model.exog,
)

bp_labels = [
    "Lagrange multiplier statistic",
    "p-value",
    "F-value",
    "F p-value",
]

bp_results = dict(zip(bp_labels, bp_test))

n = len(residuals)
centred_residuals = residuals - residuals.mean()
sum_squared_residuals = np.sum(centred_residuals ** 2)

weight_sum = 0
moran_numerator = 0.0

for i in range(n):
    neighbours = neighbour_indices[i, 1:]

    for j in neighbours:
        moran_numerator += (
            centred_residuals[i]
            * centred_residuals[j]
        )
        weight_sum += 1

morans_i = (
    (n / weight_sum)
    * (moran_numerator / sum_squared_residuals)
    if weight_sum > 0 and sum_squared_residuals > 0
    else np.nan
)

vif_features = merged_data[
    [
        "log_buffer",
        "log_res_price",
        "spatial_lag_growth",
    ]
].dropna()

vif_matrix = sm.add_constant(vif_features)

vif_data = pd.DataFrame(
    {
        "Feature": vif_matrix.columns,
        "VIF": [
            variance_inflation_factor(
                vif_matrix.values,
                i,
            )
            for i in range(vif_matrix.shape[1])
        ],
    }
)

print(
    "Breusch-Pagan p-value:",
    round(bp_results["p-value"], 4),
)
print("Residual Moran's I:", round(morans_i, 4))

vif_data

## 14. Diagnostic outputs

The diagnostic statistics are exported to Excel. Residuals are also plotted against fitted values to inspect the error structure visually.

In [ ]:
diagnostics = pd.DataFrame(
    {
        "Diagnostic": [
            "Breusch-Pagan LM statistic",
            "Breusch-Pagan p-value",
            "Residual Moran's I",
            "VIF: log GI buffer",
            "VIF: log residential price",
            "VIF: spatial lag",
        ],
        "Value": [
            bp_results["Lagrange multiplier statistic"],
            bp_results["p-value"],
            morans_i,
            vif_data.loc[
                vif_data["Feature"] == "log_buffer",
                "VIF",
            ].iloc[0],
            vif_data.loc[
                vif_data["Feature"] == "log_res_price",
                "VIF",
            ].iloc[0],
            vif_data.loc[
                vif_data["Feature"] == "spatial_lag_growth",
                "VIF",
            ].iloc[0],
        ],
    }
)

diagnostics.to_excel(
    PROJECT_DIR / "phase6_diagnostics.xlsx",
    index=False,
)

plt.figure(figsize=(9, 5))

plt.scatter(
    diagnostic_model.fittedvalues,
    residuals,
    alpha=0.7,
    s=60,
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=2,
)

plt.xlabel("Fitted value growth")
plt.ylabel("Residual")
plt.title("Residuals against fitted values")
plt.grid(True, linestyle=":", alpha=0.5)

plt.savefig(
    PROJECT_DIR / "phase6_residual_diagnostics.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

diagnostics